In [ ]:
# IMPORTING stuff including my online repo with useful tools for processing chess files (PGN)

import sys
import os, sys
!pip install chess mlflow
if os.path.exists("/kaggle/working/ludus_scacchi"):
    print("Repository already exists, skipping clone.")
    sys.path.append("/kaggle/working/ludus_scacchi")
else:
    !git clone https://github.com/paolomostardi/ludus_scacchi.git /kaggle/working/ludus_scacchi
    sys.path.append("/kaggle/working/ludus_scacchi")
from pathlib import Path
from Backend.data_pipeline import from_PGN_generate_bitboards
from Backend.data_pipeline import gm_pipeline as pipe
import pandas as pd
import numpy as np
import keras
import mlflow
import mlflow.keras

# Get MLflow credentials from Kaggle secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
mlflow_ip = user_secrets.get_secret("IP_MLFLOW")
mlflow_username = user_secrets.get_secret("USERNAME_MLFLOW")
mlflow_password = user_secrets.get_secret("PASSWORD_MLFLOW")

# Set MLflow credentials as environment variables (proper way to handle auth)
os.environ["MLFLOW_TRACKING_URI"] = f"http://{mlflow_ip}:5000"
os.environ["MLFLOW_TRACKING_USERNAME"] = mlflow_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = mlflow_password

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("ludus_cnn")
mlflow.keras.autolog(log_models=True)

print(f"Connected to remote MLflow server at: {mlflow_ip}:5000")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 77.1 MB/s eta 0:00:00:00:010:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 113.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/887.6 kB 4

2026-05-15 19:24:56.876629: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778873097.064584      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778873097.118282      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778873097.583164      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778873097.583225      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778873097.583229      57 computation_placer.cc:177] computation placer alr

In [ ]:
# MLflow credentials are now loaded in cell 0

In [4]:
# using a testing model to train faster and experiment with mlflow

from Backend.model_architecture.implementation.experimental.model_testing_mlflow import model_testing_mlflow
model = model_testing_mlflow(conv_filters=12, upsample_factor=4)

I0000 00:00:1778873119.963699      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [5]:
# deleting the folder to avoid chaos in the output
!rm -r ludus_scacchi

In [6]:
from keras.models import load_model

In [7]:
# how many cycles need to be done
# used to avoid timeout with heavier models 

MAX_N = 5
n = 5
starting_n = 0
conv_size = 32

model_name = 'lenet_baseline'
year_month = '2013-01'

mlflow_run_id = None

with mlflow.start_run(run_name=f"{model_name}_{year_month}") as run:
    mlflow_run_id = run.info.run_id
    mlflow.log_params({
        "model_name": model_name,
        "year_month": year_month,
        "max_n": MAX_N,
        "start_chunk": starting_n,
        "end_chunk": n - 1,
        "conv_size": conv_size,
        "kernel_size": (2, 2),
        "epochs_per_chunk": 3,
        "batch_size": 64
    })

    for i in range(starting_n, n):
        print("=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=")
        print("Starting training with chunk number: ", i)
        
        model.name = model_name + '_chunk_' + str(i) + '_' + year_month
        x = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '.npy')
        y = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '_y.npy')

        # leaving space for evaluation 
        if i == MAX_N - 1:
            x = x[:-10000]
            y = y[:-10000]        
        y2 = []
        for j in y:
            y2.append(j[0].flat)
        y2 = np.array(y2)
        
        history = model.fit(x, y2, batch_size=64, epochs=3)
        mlflow.log_metric("chunk_index", i, step=i)
        model.save(model.name + '.keras')
        print("saved model with name: ", model.name)
    model.save(f'{model_name}.keras')


=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  0


Epoch 1/3


I0000 00:00:1778873134.954241     159 service.cc:152] XLA service 0x7b25c40101f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778873134.954301     159 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1778873135.276464     159 cuda_dnn.cc:529] Loaded cuDNN version 91002


   42/15625 ━━━━━━━━━━━━━━━━━━━━ 59s 4ms/step - accuracy: 0.0231 - loss: 4.1873 

I0000 00:00:1778873137.919634     159 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 7298/15625 ━━━━━━━━━━━━━━━━━━━━ 30s 4ms/step - accuracy: 0.0473 - loss: 3.8604

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

if mlflow_run_id is None:
    raise RuntimeError("MLflow run was not started. Please re-run the training cell.")

# Get experiment ID from the active run
active_run = mlflow.get_run(mlflow_run_id)
experiment_id = active_run.info.experiment_id

# Save both to file
output_path = Path('/kaggle/working/mlflow_ids.txt')
output_path.write_text(f"run_id={mlflow_run_id}\nexperiment_id={experiment_id}")

print(f"Saved run_id and experiment_id to {output_path}")
print("run_id:", mlflow_run_id)
print("experiment_id:", experiment_id)